In [ ]:
import torch
import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Model

## Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.hidden_dim = torch.abs(self.in_dim - self.out_dim) / 2
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, self.hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(self.hidden_dim, self.out_dim)
        )

    def forward(self, X: torch.Tensor):
        return self.net(X)

## Decoder

In [ ]:
class Decoder(nn.Module):
    def __init__(self, *args, in_dim: int, out_dim: int, **kwargs):
        super().__init__(*args, **kwargs)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.hidden_dim = torch.abs(self.in_dim - self.out_dim) / 2
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, self.hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(self.hidden_dim, self.out_dim)
        )

    def forward(self, X: torch.Tensor):
        return self.net(X)

## Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, code_dim: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.code_dim = code_dim
        self.net = nn.Sequential(
            Encoder(self.in_dim, self.code_dim),
            Decoder(self.code_dim, self.out_dim)
        )

    def forward(self, X: torch.Tensor):
        return self.net(X)

# Configuration

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_EPOCHS = 10
BATCH_SIZE = 128

print(f'Device set to: {DEVICE}')

# Data Load

In [ ]:
training_data = FashionMNIST(
    root='train_data',
    train=True,
    download=True,
    transform=ToTensor()
)

In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

In [ ]:
data_loader = DataLoader(
    dataset=training_data,
    batch_size=BATCH_SIZE
)